# Notebook 3 — Classes (Population Segments)

A `Class` defines a named population segment: a subset of rows that share a characteristic. Once a Class is registered on a Blueprint, you can:

- **Override feature parameters** for rows in that segment — so luxury homes get a higher price distribution than entry-level homes
- **Target Influences by class** — so a pool adds 12% to a luxury home but subtracts 3% from an entry-level one (covered in Notebook 4)

Classes are evaluated against the *skeleton DataFrame* — the set of base values generated before any overrides are applied. This means a Class condition can reference any feature column that was added before the class is processed.

**Classes are not mutually exclusive.** A single row can belong to multiple classes. When two classes both override the same feature on the same row, the last-registered class wins.

In [1]:
from blueprint import Blueprint, Feature, Class
from blueprint.presets.classes import RandomClass, HighValueClass, LowValueClass, OutlierClass

import pandas as pd
import numpy as np

## Class anatomy

```python
Class(
    name,   # str — identifier used in by_class= influence dicts
    when,   # condition that selects which rows belong to this class
)
```

The `when` parameter accepts several forms — each is covered in the sections below. After construction, call `.override()` to customize feature parameters for rows in the class:

```python
Class("luxury", when=("tier", "==", "luxury"))
    .override("price", base=900_000, std=200_000)
    .override("sqft",  base=4_000,   std=1_200)
```

`.override()` returns `self`, so chaining works naturally.

## Condition type: equality `("col", "==", value)`

The most common form. Selects rows where a column equals an exact value. Works with any dtype — strings, integers, booleans, category labels.

In [2]:
bp = Blueprint(n=12, seed=42)
bp.add_feature(
    Feature("neighborhood_tier", dtype="category",
            values=["luxury", "mid", "entry"], weights=[0.25, 0.50, 0.25]),
    Feature("price", dtype=float, base=300_000, std=60_000, clip=(50_000, None)),
)

luxury = (
    Class("luxury", when=("neighborhood_tier", "==", "luxury"))
    .override("price", base=900_000, std=200_000, clip=(400_000, None))
)
entry = (
    Class("entry", when=("neighborhood_tier", "==", "entry"))
    .override("price", base=160_000, std=30_000, clip=(50_000, 250_000))
)
bp.add_class(luxury, entry)

df = bp.emit()
print(df.to_string())
print()
print("Mean price per tier:")
print(df.groupby("neighborhood_tier", observed=True)["price"].mean().round(0))

   neighborhood_tier          price
0              entry  193994.958031
1                mid  367634.472418
2              entry  151594.094615
3                mid  248442.452227
4             luxury  967046.570323
5              entry  194735.317587
6              entry  168898.818780
7              entry  165475.537730
8             luxury  852143.741026
9                mid  259144.227336
10               mid  373352.480320
11             entry  165905.020992

Mean price per tier:
neighborhood_tier
luxury    909595.0
mid       312143.0
entry     173434.0
Name: price, dtype: float64


The `mid` tier rows used the base `Feature` parameters (no class was registered for them). Luxury and entry rows each drew from their own override distributions — despite all three tiers sharing the same `price` feature definition.

## Condition type: comparison operators

Any of `>`, `>=`, `<`, `<=`, `!=` work as the second element of the condition tuple. These operate on numeric columns (and anything pandas supports comparison on).

In [3]:
bp = Blueprint(n=200, seed=42)
bp.add_feature(
    Feature("score",  dtype=int,   base=70, std=20, clip=(0, 100)),
    Feature("reward", dtype=float, base=10, std=3),
)

high_score = (
    Class("high_score", when=("score", ">", 75))
    .override("reward", base=25, std=5)
)
very_high_score = (
    Class("very_high_score", when=("score", ">", 90))
    .override("reward", base=50, std=5)
)
bp.add_class(high_score, very_high_score)

df = bp.emit()
print("score <= 75 (no class):   mean reward =", df[df.score <= 75]["reward"].mean().round(1))
print("score 76-90 (high_score): mean reward =", df[(df.score > 75) & (df.score <= 90)]["reward"].mean().round(1))
print("score > 90 (very_high):   mean reward =", df[df.score > 90]["reward"].mean().round(1))
print()
print("Condition forms demonstrated:")
print("  Class('high_score',      when=('score', '>',  75))")
print("  Class('very_high_score', when=('score', '>',  90))")
print("  Class('young',           when=('age',   '<=', 25))  # would work identically")

score <= 75 (no class):   mean reward = 10.2
score 76-90 (high_score): mean reward = 26.0
score > 90 (very_high):   mean reward = 49.8

Condition forms demonstrated:
  Class('high_score',      when=('score', '>',  75))
  Class('very_high_score', when=('score', '>',  90))
  Class('young',           when=('age',   '<=', 25))  # would work identically


Notice that rows with `score > 90` match **both** `high_score` and `very_high_score`. The class registered last (`very_high_score`) wins — so those rows get `base=50` rather than `base=25`. This is the overlap tiebreaker rule, covered in detail later in this notebook.

## Condition type: range `("col", "between", (lo, hi))`

Selects rows where `lo <= col <= hi`. Both bounds are inclusive.

In [18]:
bp = Blueprint(n=500, seed=9)
bp.add_feature(
    Feature("age",    dtype=int,   base=38, std=12, clip=(18, 70)),
    Feature("salary", dtype=float, base=65_000, std=20_000, clip=(25_000, None)),
)

# "between" condition: 30 <= age <= 50
prime_career = (
    Class("prime_career", when=("age", "between", (30, 50)))
    .override("salary", base=75_000, std=10_000, clip=(40_000, None))
)
bp.add_class(prime_career)

df = bp.emit()
print("Rows in age 30\u201350 bucket:", df["age"].between(30, 50).sum(), "/ 500")
print()
print(df[["age", "salary"]].assign(salary=df["salary"].astype(int)).head())

Rows in age 30–50 bucket: 298 / 500

   age  salary
0   28   60478
1   41   89893
2   18   61505
3   46   68845
4   52   48840


## Condition type: membership `("col", "in", [values])`

Selects rows where the column value is in a given list. Equivalent to `df[col].isin(values)`. Works with any hashable type.

In [5]:
bp = Blueprint(n=500, seed=7)
bp.add_feature(
    Feature("region",         dtype="category", values=["North", "South", "East", "West"]),
    Feature("school_dist_mi", dtype=float, base=1.5, std=0.8, clip=(0.1, 10.0)),
)

# "in" condition: region is North or East
coastal = (
    Class("coastal", when=("region", "in", ["North", "East"]))
    .override("school_dist_mi", base=0.9, std=0.4)
)
bp.add_class(coastal)

df = bp.emit()
print("Rows in coastal regions (North or East):",
      df["region"].isin(["North", "East"]).sum(), "/ 500")
print()
print("Mean school_dist_mi by region:")
print(df.groupby("region", observed=True)["school_dist_mi"].mean().round(2))

Rows in coastal regions (North or East): 250 / 500

Mean school_dist_mi by region:
region
North    0.92
South    1.51
East     0.86
West     1.47
Name: school_dist_mi, dtype: float64


## Condition type: random `("__random__", p)`

Randomly assigns a fraction `p` of rows to the class. Each row is selected independently with probability `p`, so the actual count will vary around `n × p`.

The special column name `"__random__"` signals Blueprint to use the RNG rather than look up a column in the DataFrame.

In [6]:
bp = Blueprint(n=500, seed=42)
bp.add_feature(
    Feature("user_id", dtype="id", style="sequential", start=1001),
    Feature("spend",   dtype=float, base=50, std=20, clip=(0, None)),
)

# Random 10% become VIP customers
vip = (
    Class("vip", when=("__random__", 0.10))
    .override("spend", base=200, std=50, clip=(100, None))
)
bp.add_class(vip)

df = bp.emit()

# Recover which rows are VIP by checking spend >= 100
# (non-VIP spend can theoretically reach 100 too, but the signal is clear)
high_spend_mask = df["spend"] >= 100
print("Expected VIP rows: ~50 (10% of 500)")
print()
print("All-user mean spend:  ", df["spend"].mean().round(1))
print("VIP mean spend:      ", df[high_spend_mask]["spend"].mean().round(1))
print("Non-VIP mean spend:   ", df[~high_spend_mask]["spend"].mean().round(1))
print()
print("Note: VIP membership is independent of other features \u2014")
print("any row can be selected regardless of score, age, or region.")

Expected VIP rows: ~50 (10% of 500)

All-user mean spend:   64.7
VIP mean spend:       193.7
Non-VIP mean spend:    49.1

Note: VIP membership is independent of other features —
any row can be selected regardless of score, age, or region.


## Condition type: callable `lambda df: ...`

Accepts any function that takes the skeleton DataFrame and returns a boolean array of length `n`. This is the escape hatch for conditions that can't be expressed as a simple tuple — multi-column logic, quantile thresholds, or anything else pandas can compute.

In [7]:
bp = Blueprint(n=500, seed=42)
bp.add_feature(
    Feature("age",    dtype=int,   base=38, std=12, clip=(18, 70)),
    Feature("salary", dtype=float, base=65_000, std=20_000, clip=(25_000, None)),
    Feature("bonus",  dtype=float, base=2_000, std=500, clip=(0, None)),
)

# Multi-column condition: senior AND high-earner
senior_high = (
    Class("senior_high_earner",
          when=lambda df: (df["age"] >= 55) & (df["salary"] >= 80_000))
    .override("bonus", base=15_000, std=3_000)
)
bp.add_class(senior_high)

df = bp.emit()

mask = (df["age"] >= 55) & (df["salary"] >= 80_000)
print("Callable condition: age >= 55 AND salary >= 80,000")
print("Rows matched:", mask.sum())
print()
print("Control group mean bonus:       ", df[~mask]["bonus"].mean().round(0))
print("Senior high-earner mean bonus: ", df[mask]["bonus"].mean().round(0))

Callable condition: age >= 55 AND salary >= 80,000
Rows matched: 5

Control group mean bonus:        1999.0
Senior high-earner mean bonus:  14898.0


The lambda receives the full skeleton DataFrame, so you can use any pandas expression — including `.quantile()`, `.rolling()`, string methods, and comparisons between columns.

## Overriding feature parameters with `.override()`

`.override(feature_name, **kwargs)` merges the provided kwargs into the base `Feature`'s parameters. Only the keys you supply are replaced — everything else (dtype, unspecified clip bounds, modifiers) is inherited from the original feature.

The most common override targets:

| Override target | Purpose |
|---|---|
| `base`, `std` | Shift the mean/spread for this segment |
| `clip` | Tighten or widen the hard bounds |
| `p` | Change `True` probability for bool features |
| `values`, `weights` | Replace category distribution for this segment |

In [8]:
bp = Blueprint(n=1000, seed=42)
bp.add_feature(
    Feature("plan",  dtype="category", values=["free", "basic", "pro"], weights=[5, 3, 2]),
    Feature("age",   dtype=int,   base=35, std=10, clip=(18, 65)),
    Feature("spend", dtype=float, base=50, std=20, clip=(0, None)),
)

free_users = (
    Class("free_users", when=("plan", "==", "free"))
    .override("spend", base=0, std=5, clip=(0, None))   # very low spend, near zero
    .override("age",   base=28, std=8)                  # younger demographic
)
pro_users = (
    Class("pro_users", when=("plan", "==", "pro"))
    .override("spend", base=200, std=50, clip=(100, None))  # high committed spend
    .override("age",   base=42, std=8)                      # older professional
)
# basic users: no class — they use the base Feature parameters

bp.add_class(free_users, pro_users)

df = bp.emit()
print(df.groupby("plan", observed=True)[["age", "spend"]].mean().round(1))
print()
print("Row counts \u2014 free:", (df.plan == "free").sum(),
      " basic:", (df.plan == "basic").sum(),
      " pro:", (df.plan == "pro").sum())

        age  spend
plan              
free   28.5    2.0
basic  35.4   51.3
pro    41.2  192.2

Row counts — free: 503  basic: 300  pro: 197


Notice that `basic` users were never given a class — they simply got the base feature distributions. You only need to register a class if you want to override something, or if you plan to reference that class name in an `Influence` (see Notebook 4).

### `.override()` is chainable and returns `self`

In [9]:
luxury = (
    Class("luxury", when=("tier", "==", "luxury"))
    .override("price",     base=900_000, std=200_000, clip=(400_000, None))
    .override("sqft",      base=4_000,   std=1_200)
    .override("has_pool",  p=0.75)
    .override("has_garage",p=0.95)
)

print("luxury overrides:", luxury.overrides)
print()
print("Number of overrides:", len(luxury.overrides))

# Confirm chainability
c = Class("test", when=("x", "==", 1))
result = c.override("y", base=10)
print(".override() returns the same object:", result is c)

luxury overrides: {'price': {'base': 900000, 'std': 200000, 'clip': (400000, None)}, 'sqft': {'base': 4000, 'std': 1200}, 'has_pool': {'p': 0.75}, 'has_garage': {'p': 0.95}}

Number of overrides: 4
.override() returns the same object: True


## Class overlap: registration order as tiebreaker

When two classes both match the same row and both override the same feature, the **last-registered class wins**. Classes are applied in registration order — later registrations overwrite earlier ones on contested rows.

This means you should register **more specific classes after more general ones**. The specific class will override the general one where they overlap.

In [10]:
bp = Blueprint(n=200, seed=42)
bp.add_feature(
    Feature("score",  dtype=int,   base=70, std=20, clip=(0, 100)),
    Feature("reward", dtype=float, base=10, std=3),
)

# Register general class first, specific class second
high_score = (
    Class("high_score",      when=("score", ">", 75))
    .override("reward", base=25, std=5)
)
very_high_score = (
    Class("very_high_score", when=("score", ">", 90))
    .override("reward", base=50, std=5)
)

bp.add_class(high_score, very_high_score)  # very_high_score is last — it wins on overlap

df = bp.emit()
print("Score buckets:")
print("score <= 75 (no class):    mean reward =",
      df[df.score <= 75]["reward"].mean().round(1))
print("score 76-90 (high_score):  mean reward =",
      df[(df.score > 75) & (df.score <= 90)]["reward"].mean().round(1))
print("score > 90  (very_high):   mean reward =",
      df[df.score > 90]["reward"].mean().round(1))
print()
print("Rows with score > 90:", (df.score > 90).sum())
print("These rows match BOTH classes; very_high_score (registered last) wins.")

Score buckets:
score <= 75 (no class):    mean reward = 10.2
score 76-90 (high_score):  mean reward = 26.0
score > 90  (very_high):   mean reward = 49.8

Rows with score > 90: 21
These rows match BOTH classes; very_high_score (registered last) wins.


### Reversing the order inverts the winner

In [11]:
bp2 = Blueprint(n=200, seed=42)
bp2.add_feature(
    Feature("score",  dtype=int,   base=70, std=20, clip=(0, 100)),
    Feature("reward", dtype=float, base=10, std=3),
)
# Reversed registration order — high_score is now last
bp2.add_class(very_high_score, high_score)

df2 = bp2.emit()
print("With very_high_score registered FIRST:")
print("score > 90 mean reward =", df2[df2.score > 90]["reward"].mean().round(1),
      "\u2190 high_score (the later one) won instead")

With very_high_score registered FIRST:
score > 90 mean reward = 26.1 ← high_score (the later one) won instead


**The rule:** register classes from most general to most specific. The specific class should always be last so it wins when rows fall in both buckets.

Non-overlapping features are unaffected — if `high_score` overrides `reward` and `very_high_score` overrides `badge_level`, both overrides apply independently regardless of order.

## `describe()` shows registered classes

`Blueprint.describe()` summarises all registered classes — their `when` condition and which features they override.

In [12]:
bp = Blueprint(n=100, seed=42)
bp.add_feature(
    Feature("tier",     dtype="category", values=["gold", "silver", "bronze"], weights=[1, 3, 6]),
    Feature("score",    dtype=int,   base=70,   std=15,   clip=(0, 100)),
    Feature("discount", dtype=float, base=0.05, std=0.02, clip=(0, 0.5)),
)

gold_class = (
    Class("gold_members", when=("tier", "==", "gold"))
    .override("discount", base=0.20, std=0.05)
    .override("score",    base=85,   std=10)
)
vip_random = Class("vip_bonus", when=("__random__", 0.10))

bp.add_class(gold_class, vip_random)
bp.describe()

Blueprint(n=100, seed=42)

Features (3):
  tier                 category
  score                <class 'int'> base=70, std=15, clip=(0, 100)
  discount             <class 'float'> base=0.05, std=0.02, clip=(0, 0.5)

Classes (2):
  gold_members         when: tier == gold, overrides: ['discount', 'score']
  vip_bonus            when: random p=0.1


## Preset classes

Blueprint ships four preset factories for common segmentation patterns. Each returns a fully-configured `Class` object that accepts `.override()` chaining.

```python
from blueprint.presets.classes import RandomClass, HighValueClass, LowValueClass, OutlierClass
```

### `RandomClass(name, p)`

Shorthand for `Class(name, when=("__random__", p))`. Use when you want to flag a random fraction of the population without basing it on any feature value.

In [13]:
# RandomClass — equivalent to Class(name, when=("__random__", p))
manual_review = RandomClass("manual_review", p=0.05)

print("RandomClass is equivalent to Class(name, when=(\"__random__\", p)):")
print("  manual_review.when ==", manual_review.when)
print()
print("Expected ~50 flagged rows. Actual: varies per run (probabilistic).")

RandomClass is equivalent to Class(name, when=("__random__", p)):
  manual_review.when == ('__random__', 0.05)

Expected ~50 flagged rows. Actual: varies per run (probabilistic).


### `HighValueClass(name, feature, top_pct)`

Selects the top `top_pct` fraction of rows by a numeric feature. Implemented as a callable condition using `.quantile()`.

In [14]:
bp = Blueprint(n=1000, seed=99)
bp.add_feature(
    Feature("income",       dtype=float, base=55_000,  std=20_000, clip=(15_000, None)),
    Feature("credit_score", dtype=int,   base=680,     std=80,     clip=(300, 850)),
    Feature("loan_amount",  dtype=float, base=200_000, std=60_000, clip=(10_000, None)),
)

# Top 10% by income get higher loan amounts
high_earners = (
    HighValueClass("high_earners", feature="income", top_pct=0.10)
    .override("loan_amount", base=500_000, std=100_000)
)

# Bottom 20% by credit_score get capped loan amounts
poor_credit = (
    LowValueClass("poor_credit", feature="credit_score", bottom_pct=0.20)
    .override("loan_amount", base=80_000, std=20_000, clip=(10_000, 150_000))
)

bp.add_class(high_earners, poor_credit)
df = bp.emit()

top10_mask = df["income"] >= df["income"].quantile(0.90)
bot20_mask = df["credit_score"] <= df["credit_score"].quantile(0.20)

print("HighValueClass (top 10% income):")
print("  count:", top10_mask.sum())
print("  mean loan:", df[top10_mask]["loan_amount"].mean().round(0))
print()
print("LowValueClass (bottom 20% credit_score):")
print("  count:", bot20_mask.sum())
print("  mean loan:", df[bot20_mask]["loan_amount"].mean().round(0))
print()
print("Overall mean loan:", df["loan_amount"].mean().round(0))

HighValueClass (top 10% income):
  count: 100
  mean loan: 444312.0

LowValueClass (bottom 20% credit_score):
  count: 203
  mean loan: 82197.0

Overall mean loan: 202486.0


### `LowValueClass(name, feature, bottom_pct)`

Mirror of `HighValueClass` — selects the bottom `bottom_pct` fraction. Already shown above alongside `HighValueClass`.

### `OutlierClass(name, p, features, magnitude)`

`OutlierClass` is a random class (probability `p`) that stores metadata about which features it affects and by how much. The metadata (`.outlier_features`, `._outlier_magnitude`) is available for downstream use, but the override values themselves are not applied automatically — you supply them via `.override()` as usual.

In [15]:
bp = Blueprint(n=500, seed=42)
bp.add_feature(
    Feature("revenue", dtype=float, base=50_000, std=10_000, clip=(10_000, None)),
    Feature("units",   dtype=int,   base=100,    std=20,     clip=(1, None)),
)

# OutlierClass: 3% of rows are anomalies with ~4× normal revenue
# The features/magnitude metadata documents intent; the override does the actual work
anomaly = (
    OutlierClass("anomaly", p=0.03, features=["revenue", "units"], magnitude=4.0)
    .override("revenue", base=200_000, std=30_000)
    .override("units",   base=400,     std=50)
)
bp.add_class(anomaly)

df = bp.emit()

print("Total rows:", len(df))
print("Outlier rows (revenue > 150k):", (df.revenue > 150_000).sum())
print()
print("Normal mean revenue: ", df[df.revenue <= 150_000]["revenue"].mean().round(0))
print("Outlier mean revenue:", df[df.revenue > 150_000]["revenue"].mean().round(0))
print()
print("OutlierClass metadata:")
print("  _outlier_features: ", anomaly._outlier_features)
print("  _outlier_magnitude:", anomaly._outlier_magnitude)
print("  when:              ", anomaly.when)

Total rows: 500
Outlier rows (revenue > 150k): 10

Normal mean revenue:  49901.0
Outlier mean revenue: 212552.0

OutlierClass metadata:
  _outlier_features:  ['revenue', 'units']
  _outlier_magnitude: 4.0
  when:               ('__random__', 0.03)


## Putting it together: a multi-class blueprint

Here is a realistic example combining all condition types. A loan application dataset with four distinct population segments.

In [16]:
bp = Blueprint(n=800, seed=7)
bp.add_feature(
    Feature("applicant_id",  dtype="id",    style="prefixed", prefix="APP-", padding=4),
    Feature("income",        dtype=float,   base=55_000, std=20_000, clip=(15_000, None)),
    Feature("credit_score",  dtype=int,     base=680,    std=80,     clip=(300, 850)),
    Feature("loan_amount",   dtype=float,   base=200_000, std=60_000, clip=(10_000, None)),
)

# Callable: top 15% by income → higher loans
high_earners = (
    Class("high_earners",
          when=lambda df: df["income"] >= df["income"].quantile(0.85))
    .override("loan_amount", base=450_000, std=80_000)
)
# Callable: bottom 15% by credit_score → capped loans
poor_credit = (
    Class("poor_credit",
          when=lambda df: df["credit_score"] <= df["credit_score"].quantile(0.15))
    .override("loan_amount", base=70_000, std=15_000, clip=(10_000, 120_000))
)
# Between: mid-credit band (620-720) → moderate loans
mid_credit = (
    Class("mid_credit", when=("credit_score", "between", (620, 720)))
    .override("loan_amount", base=160_000, std=40_000)
)
# Random: 5% flagged for extra review (no override — just a label for influences)
flagged = RandomClass("random_flag", p=0.05)

# Register general → specific so specific classes win on overlap
bp.add_class(mid_credit, poor_credit, high_earners, flagged)

bp.describe()
print()
df = bp.emit()
df.head().round(2)

Blueprint(n=800, seed=7)

Features (4):
  applicant_id         id
  income               <class 'float'> base=55000, std=20000, clip=(15000, None)
  credit_score         <class 'int'> base=680, std=80, clip=(300, 850)
  loan_amount          <class 'float'> base=200000, std=60000, clip=(10000, None)

Classes (4):
  mid_credit           when: credit_score between (620, 720), overrides: ['loan_amount']
  poor_credit          when: <callable>, overrides: ['loan_amount']
  high_earners         when: <callable>, overrides: ['loan_amount']
  random_flag          when: random p=0.05



,applicant_id,income,credit_score,loan_amount
0,APP-0001,55024.60,736,98534.19
1,APP-0002,60974.91,671,134912.57
2,APP-0003,49517.24,709,127571.92
3,APP-0004,37188.16,688,109697.03
4,APP-0005,45906.58,731,129958.42


## Summary

| Condition form | Syntax | Selects |
|---|---|---|
| Equality | `("col", "==", value)` | `df[col] == value` |
| Inequality | `("col", "!=", value)` | `df[col] != value` |
| Comparison | `("col", ">", value)` etc. | `df[col] > value` (or `>=`, `<`, `<=`) |
| Range | `("col", "between", (lo, hi))` | `lo <= df[col] <= hi` |
| Membership | `("col", "in", [v1, v2, ...])` | `df[col].isin(...)` |
| Random | `("__random__", p)` | each row independently with probability `p` |
| Callable | `lambda df: ...` | any boolean expression over the full DataFrame |

**Key rules:**
- Classes are not mutually exclusive — a row can match multiple classes
- When two classes override the same feature on the same row, the last-registered class wins
- Register from most general to most specific so specific overrides always apply
- A class with no `.override()` calls is still useful as a label target for `by_class=` in Influences

**What's next:**
- **Notebook 4** — Influences: express causal relationships between columns, and use `by_class=` to vary the effect by population segment
- **Notebook 5** — The dependency DAG: how Blueprint resolves evaluation order when features influence each other